# 17. 판정요지 불용어 그룹별 제거 효과 및 HS 코드 예측 평가

16번에서는 판정요지 전체에 22개 불용어를 적용했을 때 검색 지표가 개선되는 것을 확인했습니다. 이번에는 어떤 단어 그룹이 개선에 기여하는지 확인하고, 검색 결과를 HS 코드 예측에 활용했을 때 Accuracy와 F1-score도 평가합니다.

평가 조건:
- 신규 제품의 질의는 제품 속성 5개만 사용
- 과거 사례 검색 문서는 제품 속성 5개 + 판정요지
- 동일 제품 속성 그룹이 Fold 사이에 중복되지 않도록 4-fold StratifiedGroupKFold 적용
- `intfloat/multilingual-e5-base` 사용
- 검색 순위 지표와 Top-1 / 유사도 가중 Top-3·Top-5 HS 코드 예측 성능을 함께 평가

불용어 그룹은 의미적 역할을 기준으로 구성합니다. 각 그룹 제거 조건은 전체 불용어 목록에서 해당 그룹만 제외하고 나머지 불용어를 적용합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy

In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data = pd.read_csv(BASE+cases_file).fillna('')
INPUT_COLUMNS = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET = 'HS Code'
SUMMARY = 'Preprocessed Ruling Summary'
for col in INPUT_COLUMNS+[TARGET,SUMMARY]:
    data[col] = data[col].astype(str).replace('nan','').fillna('').str.strip()
data = data[data[TARGET].ne('')].copy()
data['query_text'] = data[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
data['duplicate_group'] = data[INPUT_COLUMNS].agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data.groupby(TARGET).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(TARGET)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[TARGET].isin(labels)].copy().reset_index(drop=True)
data[TARGET] = data[TARGET].astype(str)

STOPWORD_GROUPS = {
    '일반_문서표현': {'것','경우','등','따라서','및','수','이상','함'},
    '관세_규정표현': {'관세율표','규정','기재','관세','표'},
    '분류_품목표현': {'분류','소호','호','해당','제품','물품'},
    '사례_설명표현': {'본건','본품','설명'}
}
ALL_STOPWORDS = set().union(*STOPWORD_GROUPS.values())
kiwi = Kiwi()

def remove_morphs(text,stopwords):
    text = str(text)
    chars = list(text)
    for token in kiwi.tokenize(text):
        if token.form in stopwords:
            for j in range(token.start,token.start+token.len): chars[j] = ''
    return ''.join(chars)

print('평가 사례:',len(data),'| HS 코드:',data[TARGET].nunique())
print('판정요지 결측:',int(data[SUMMARY].eq('').sum()))
print('불용어 수:',len(ALL_STOPWORDS))

## 1. 비교 조건

- `product_only`: 제품 속성만 포함한 과거 사례 문서
- `ruling_original`: 제품 속성 + 원본 판정요지
- `ruling_all_stopwords`: 전체 불용어 적용
- `minus_<그룹>`: 전체 불용어에서 해당 그룹만 제외한 상태로 전처리

따라서 `minus_일반_문서표현`의 결과가 전체 불용어 적용보다 좋다면, 해당 그룹을 제거 대상에서 제외하는 것이 유리할 수 있습니다.

In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
cv = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)

def encode_docs(texts):
    return model.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_query(text):
    return model.encode(['query: '+str(text)],normalize_embeddings=True,convert_to_numpy=True)[0]
def weighted_vote(scores,labels,k):
    order = np.argsort(scores)[::-1][:k]
    votes = {}
    for ix in order:
        label = str(labels[ix])
        votes[label] = votes.get(label,0.0)+max(float(scores[ix]),0.0)
    return sorted(votes.items(),key=lambda x:(-x[1],x[0]))[0][0]
def retrieval_metrics(scores,labels,true_label):
    order = np.argsort(scores)[::-1]
    relevant = np.asarray(labels,dtype=str)==str(true_label)
    ranked = relevant[order]
    hits = np.where(ranked)[0]
    mrr = 1/(hits[0]+1) if len(hits) else 0.0
    gains = ranked[:10].astype(float)
    dcg = float((gains/np.log2(np.arange(2,len(gains)+2))).sum())
    ideal_n = min(int(relevant.sum()),10)
    idcg = float((np.ones(ideal_n)/np.log2(np.arange(2,ideal_n+2))).sum()) if ideal_n else 0.0
    return {'mrr':mrr,'ndcg@10':dcg/idcg if idcg else 0.0,'hit@5':float(ranked[:5].any()),'hit@10':float(ranked[:10].any())}

## 2. Fold별 검색 및 예측

Top-1은 가장 유사한 과거 사례의 HS 코드를 그대로 예측합니다. Top-3·Top-5는 상위 사례들의 cosine similarity를 가중치로 사용해 HS 코드를 투표합니다. 검색 결과가 곧 법적 판정은 아니며, 과거 사례 기반 예측 성능을 평가하는 실험입니다.

In [ ]:
result_rows,fold_rows = [],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[TARGET],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    train_labels = train[TARGET].to_numpy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    variants = {'product_only':None,'ruling_original':set(),'ruling_all_stopwords':ALL_STOPWORDS}
    for group_name,group_words in STOPWORD_GROUPS.items():
        variants['minus_'+group_name] = ALL_STOPWORDS-group_words
    doc_embeddings = {}
    for method,stopwords in variants.items():
        if method=='product_only':
            texts = train['query_text'].tolist()
        elif method=='ruling_original':
            texts = (train['query_text']+' '+train[SUMMARY]).str.strip().tolist()
        else:
            clean_summary = train[SUMMARY].map(lambda x:remove_morphs(x,stopwords))
            texts = (train['query_text']+' '+clean_summary).str.strip().tolist()
        doc_embeddings[method] = encode_docs(texts)
    for idx,row in test.iterrows():
        q = encode_query(row['query_text'])
        for method,emb in doc_embeddings.items():
            scores = emb @ q
            met = retrieval_metrics(scores,train_labels,row[TARGET])
            order = np.argsort(scores)[::-1]
            result_rows.append({
                'fold':fold,'row_index':int(idx),'true_hs':row[TARGET],'method':method,**met,
                'pred_top1':train_labels[order[0]],
                'pred_top3':weighted_vote(scores,train_labels,3),
                'pred_top5':weighted_vote(scores,train_labels,5)
            })
    print('Fold',fold,'완료')

results = pd.DataFrame(result_rows)
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
print('Fold 그룹 중복 합계:',int(fold_audit['group_overlap'].sum()))

## 3. 검색 성능 비교

검색 성능은 MRR, nDCG@10, Hit@5, Hit@10으로 비교합니다. 검색 성능이 높더라도 HS 코드 예측 성능이 반드시 높아지는 것은 아니므로 다음 절에서 Accuracy와 F1-score를 별도로 확인합니다.

In [ ]:
search_metrics = ['mrr','ndcg@10','hit@5','hit@10']
search_summary = results.groupby('method')[search_metrics].agg(['mean','std']).round(4)
display(search_summary)

## 4. HS 코드 예측 성능

Accuracy, Macro-F1, Weighted-F1을 Top-1·Top-3·Top-5 예측별로 계산합니다. Macro-F1은 각 HS 코드의 성능을 동일한 비중으로 평가하고, Weighted-F1은 사례 수가 많은 HS 코드에 더 큰 비중을 둡니다.

In [ ]:
prediction_rows = []
for method,g in results.groupby('method'):
    for k in [1,3,5]:
        pred_col = f'pred_top{k}'
        prediction_rows.append({
            'method':method,'prediction':'Top-'+str(k),
            'accuracy':accuracy_score(g['true_hs'],g[pred_col]),
            'macro_f1':f1_score(g['true_hs'],g[pred_col],average='macro',zero_division=0),
            'weighted_f1':f1_score(g['true_hs'],g[pred_col],average='weighted',zero_division=0)
        })
prediction_summary = pd.DataFrame(prediction_rows)
display(prediction_summary.round(4))
display(results[results['method'].eq('ruling_all_stopwords')].groupby('fold').apply(lambda g:pd.Series({'accuracy_top1':accuracy_score(g.true_hs,g.pred_top1),'macro_f1_top1':f1_score(g.true_hs,g.pred_top1,average='macro',zero_division=0)}),include_groups=False).round(4))

## 5. 불용어 그룹별 기여도

전체 불용어 적용 조건과 각 그룹을 제거 대상에서 제외한 조건을 비교합니다. 특정 그룹을 제외했을 때 성능이 상승하면 해당 그룹은 불용어로 제거하지 않는 편이 유리할 가능성이 있습니다.

In [ ]:
base = prediction_summary[(prediction_summary.method=='ruling_all_stopwords')&(prediction_summary.prediction=='Top-1')].iloc[0]
ablation_rows = []
for method in variants:
    row = prediction_summary[(prediction_summary.method==method)&(prediction_summary.prediction=='Top-1')].iloc[0]
    ablation_rows.append({'method':method,'accuracy':row.accuracy,'macro_f1':row.macro_f1,'weighted_f1':row.weighted_f1,'accuracy_diff_vs_all':row.accuracy-base.accuracy,'macro_f1_diff_vs_all':row.macro_f1-base.macro_f1,'weighted_f1_diff_vs_all':row.weighted_f1-base.weighted_f1})
ablation_summary = pd.DataFrame(ablation_rows)
display(ablation_summary.round(4))
display(results.groupby('method')[search_metrics].mean().round(4))

## 6. 결과 저장

- `17_group_ablation_predictions.csv`: 제품별 검색 지표 및 Top-k 예측
- `17_group_ablation_search_summary.csv`: 검색 지표 요약
- `17_group_ablation_prediction_summary.csv`: Accuracy·F1 요약
- `17_group_ablation_summary.csv`: 불용어 그룹별 차이
- `17_fold_audit.csv`: Fold별 데이터 분리 점검

In [ ]:
results.to_csv('17_group_ablation_predictions.csv',index=False,encoding='utf-8-sig')
search_summary.reset_index().to_csv('17_group_ablation_search_summary.csv',index=False,encoding='utf-8-sig')
prediction_summary.to_csv('17_group_ablation_prediction_summary.csv',index=False,encoding='utf-8-sig')
ablation_summary.to_csv('17_group_ablation_summary.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('17_fold_audit.csv',index=False,encoding='utf-8-sig')
print('17번 결과 저장 완료')